# PyTorch Essentials

Notebook for **Chapter 5** of the *dl* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dl/ch05.html

Tip: change a number, predict the output, then run the cell again.

## 5.1 Tensors and Autograd

In [ ]:
import torch
a = torch.tensor([[1., 2.], [3., 4.]])      # 2x2 float32 tensor
b = torch.ones(2, 2)
c = a @ b + 1                                # matrix multiply, then add
w = torch.tensor(2.0, requires_grad=True)    # ask PyTorch to track gradients
loss = (w * 3 - 1) ** 2                      # loss = (3w - 1)^2
loss.backward()                              # fills w.grad
print(w.grad)                                # d loss/dw = 2*(3w-1)*3 = 30

In [ ]:
import math
class Value:
    def __init__(s, data, parents=(), back=None):
        s.data = float(data); s.grad = 0.0; s._p = parents; s._b = back
    def __add__(s, o):
        o = o if isinstance(o, Value) else Value(o); out = Value(s.data + o.data, (s, o))
        def b(): s.grad += out.grad; o.grad += out.grad
        out._b = b; return out
    def __mul__(s, o):
        o = o if isinstance(o, Value) else Value(o); out = Value(s.data * o.data, (s, o))
        def b(): s.grad += o.data * out.grad; o.grad += s.data * out.grad
        out._b = b; return out
    def sigmoid(s):
        v = 1/(1 + math.exp(-s.data)); out = Value(v, (s,))
        def b(): s.grad += v*(1-v) * out.grad
        out._b = b; return out
    def log(s):
        out = Value(math.log(s.data), (s,))
        def b(): s.grad += out.grad / s.data
        out._b = b; return out
    def backward(s):
        order, seen = [], set()
        def topo(v):
            if id(v) not in seen:
                seen.add(id(v))
                for p in v._p: topo(p)
                order.append(v)
        topo(s); s.grad = 1.0
        for v in reversed(order):
            if v._b: v._b()

x = [1., 2.]
W1 = [[Value(.1), Value(.2)], [Value(.3), Value(.4)]]; b1 = [Value(.1), Value(-.1)]
W2 = [Value(.5), Value(-.6)]; b2 = Value(.2)
h = [(W1[i][0]*x[0] + W1[i][1]*x[1] + b1[i]).sigmoid() for i in range(2)]
p = (W2[0]*h[0] + W2[1]*h[1] + b2).sigmoid()
L = p.log() * (-1)
L.backward()
print("L", round(L.data,4), "dW1[0][1]", round(W1[0][1].grad,6),
      "dW2", [round(w.grad,4) for w in W2], "db2", round(b2.grad,4))

L 0.6519 dW1[0][1] -0.109579 dW2 [-0.3092, -0.3502] db2 -0.479


## 5.2nn.Module,Dataset,DataLoaderand the Training Loop

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class Moons(Dataset):
    def __init__(self, X, y): self.X = torch.tensor(X, dtype=torch.float32); self.y = torch.tensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
    def forward(self, x): return self.body(x)        # returns logits, no softmax

model = Net()
loader = DataLoader(Moons(X, y), batch_size=32, shuffle=True)
loss_fn = nn.CrossEntropyLoss()                        # expects logits
opt = torch.optim.AdamW(model.parameters(), lr=1e-2, weight_decay=0.01)

for epoch in range(100):
    model.train()
    for xb, yb in loader:
        opt.zero_grad()                  # 1. clear old gradients
        loss = loss_fn(model(xb), yb)    # 2. forward + loss
        loss.backward()                  # 3. backward
        opt.step()                       # 4. update weights

In [ ]:
model.eval()
correct = 0
with torch.no_grad():
    for xb, yb in loader:
        correct += (model(xb).argmax(1) == yb).sum().item()
print(correct / len(loader.dataset))

In [ ]:
exec(open("ch2_mlp.py").read())     # data X, y and class MLP from Chapter 2, saved to a file
import math
N = len(X)
print("batches per epoch", math.ceil(N/32), "last batch", N - 32*(math.ceil(N/32)-1))
m = MLP([2,16,2], seed=0); rng = np.random.default_rng(0)
for epoch in range(100):
    idx = rng.permutation(N)
    for s in range(0, N, 32):
        b = idx[s:s+32]
        m.forward(X[b]); gW, gb = m.backward(y[b])
        for k in range(len(m.W)):
            m.W[k] -= 0.1*gW[k]; m.b[k] -= 0.1*gb[k]
    if epoch % 20 == 0 or epoch == 99:
        print(epoch, round(m.loss(X,y),4), "acc", (m.forward(X).argmax(1) == y).mean())

batches per epoch 7 last batch 8
0 0.4375 acc 0.805
20 0.2505 acc 0.89
40 0.2279 acc 0.9
60 0.207 acc 0.915
80 0.1835 acc 0.92
99 0.1669 acc 0.935


## 5.3 GPUs, Mixed Precision, Accumulation, Checkpoints and Reproducibility

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
for xb, yb in loader:
    xb, yb = xb.to(device), yb.to(device)       # data must be on the same device

In [ ]:
print("fp16 max", np.finfo(np.float16).max, "fp16 tiny subnormal", np.finfo(np.float16).smallest_subnormal)
print("70000 -> fp16", np.float16(70000))
print("1e-8 -> fp16", np.float16(1e-8), " x1024 ->", np.float16(1e-8*1024),
      "unscaled back", float(np.float16(1e-8*1024))/1024)
print("1+0.0004 in fp16", np.float16(1)+np.float16(0.0004), "in fp32", np.float32(1)+np.float32(0.0004))
def bf16(a):
    a = np.float32(a); return (a.view(np.uint32) & np.uint32(0xFFFF0000)).view(np.float32)
for v in [1001.3, 70000., 1e-8]:
    print(v, "fp16", np.float16(v), "bf16", float(bf16(v)))

fp16 max 6.55e+04 fp16 tiny subnormal 6e-08
70000 -> fp16 inf
1e-8 -> fp16 0.0  x1024 -> 1.025e-05 unscaled back 1.0011717677116394e-08
1+0.0004 in fp16 1.0 in fp32 1.0004
1001.3 fp16 1.0015e+03 bf16 1000.0
70000.0 fp16 inf bf16 69632.0
1e-08 fp16 0.0 bf16 9.953510016202927e-09


In [ ]:
scaler = torch.amp.GradScaler()                   # needed for fp16; bf16 usually does not need it
for xb, yb in loader:
    opt.zero_grad()
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        loss = loss_fn(model(xb.to("cuda")), yb.to("cuda"))
    scaler.scale(loss).backward()
    scaler.step(opt); scaler.update()

In [ ]:
r = np.random.default_rng(0); X = r.normal(size=(8,3)); y = r.normal(size=8); w = r.normal(size=3)
g_full = 2*X.T@(X@w - y)/8
g_acc = np.zeros(3)
for i in range(4):
    xb, yb = X[2*i:2*i+2], y[2*i:2*i+2]
    g_acc += 2*xb.T@(xb@w - yb)/2/4
print("full", g_full, "accum", g_acc, "equal", np.allclose(g_full, g_acc))

full [-0.1139  0.6338 -0.4902] accum [-0.1139  0.6338 -0.4902] equal True


## 5.4 Seeing Inside the Machine: The Loop, the Graph, Memory Tricks, Compile and Resume

In [ ]:
# a tiny reverse-mode autodiff: every node remembers its parents and the local derivative to each
class V:
    def __init__(s, val, parents=()):
        s.val, s.parents, s.grad = val, parents, 0.0     # parents: list of (node, local derivative)
    def __add__(s, o): return V(s.val + o.val, [(s, 1.0), (o, 1.0)])
    def __mul__(s, o): return V(s.val * o.val, [(s, o.val), (o, s.val)])
def backward(out):
    order, seen = [], set()
    def visit(n):
        if id(n) in seen: return
        seen.add(id(n))
        for p, _ in n.parents: visit(p)
        order.append(n)
    visit(out); out.grad = 1.0
    for n in reversed(order):                            # children before parents
        for p, d in n.parents: p.grad += d * n.grad      # += : gradients from several paths ADD
a, b, c = V(2.0), V(3.0), V(1.0)
v = a*b + c
f = v*v
backward(f)
print("f =", f.val, " df/da =", a.grad, " df/db =", b.grad, " df/dc =", c.grad)
a, b = V(2.0), V(3.0)
g = a*b + a*a
backward(g)
print("g =", g.val, " dg/da =", a.grad, "(b + 2a = 7: two paths add)", " dg/db =", b.grad)
# gradient checkpointing arithmetic: L layers, keep every k-th activation, recompute the rest
for L in (16, 64, 256):
    k = int(L**0.5)
    kept = L//k + k           # stored checkpoints + one segment recomputed at a time
    print(f"L={L:4d}: store all = {L:4d} units, checkpoint every {k:2d}: peak about {kept:3d} units, extra compute = 1 more forward pass")

f = 49.0  df/da = 42.0  df/db = 28.0  df/dc = 14.0
g = 10.0  dg/da = 7.0 (b + 2a = 7: two paths add)  dg/db = 2.0
L=  16: store all =   16 units, checkpoint every  4: peak about   8 units, extra compute = 1 more forward pass
L=  64: store all =   64 units, checkpoint every  8: peak about  16 units, extra compute = 1 more forward pass
L= 256: store all =  256 units, checkpoint every 16: peak about  32 units, extra compute = 1 more forward pass


In [ ]:
import torch
a = torch.tensor(2.0, requires_grad=True); b = torch.tensor(3.0, requires_grad=True); c = torch.tensor(1.0, requires_grad=True)
f = (a*b + c)**2
f.backward()
print(a.grad, b.grad, c.grad)    # expected: tensor(42.) tensor(28.) tensor(14.)

In [ ]:
import torch
model = MyModel().to(device)
model = torch.compile(model)           # first call is slow: it compiles. Later calls are faster.
for xb, yb in loader:
    loss = loss_fn(model(xb.to(device)), yb.to(device))
    loss.backward(); opt.step(); opt.zero_grad()

In [ ]:
import torch
from torch.profiler import profile, ProfilerActivity
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True) as prof:
    for step, (xb, yb) in enumerate(loader):
        loss = loss_fn(model(xb.to(device)), yb.to(device)); loss.backward(); opt.step(); opt.zero_grad()
        if step == 5: break
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=8))

In [ ]:
import torch
def save(path, model, opt, sched, step):
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "step": step, "rng": torch.get_rng_state()}, path)
def load(path, model, opt, sched):
    ck = torch.load(path, map_location="cpu")
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
    torch.set_rng_state(ck["rng"]); return ck["step"]

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
x = torch.randn(512, 2)
y = ((x[:, 0] * x[:, 1]) > 0).float().unsqueeze(1)          # label: are the two signs equal?
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
loss_fn = nn.BCEWithLogitsLoss()
x, y = x.to(device), y.to(device)
for epoch in range(200):
    opt.zero_grad(); loss = loss_fn(model(x), y); loss.backward(); opt.step()
    if epoch % 50 == 0: print(epoch, round(loss.item(), 4))
acc = ((model(x) > 0).float() == y).float().mean().item()
print("train accuracy:", round(acc, 3))

## Chapter recap

## Practice set

**1.** (easy) `w = 2`, loss `(3w − 1)²`. What is `d loss / dw`? **Answer:** `2(3w − 1)·3 = 2·5·3 = 30`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dl/ch05.html